In [2]:
import pandas as pd
# Charger le dataset original
df = pd.read_csv("../data/tickets.csv")
# Créer une copie de travail
df_clean = df.copy()
# Vérifier le chargement
print("Dataset original :", df.shape)
print("Dataset de travail :", df_clean.shape)


Dataset original : (4000, 7)
Dataset de travail : (4000, 7)


## 2. Remove Exact Duplicates
Keep only one copy of each ticket description.

Objective: Avoid duplicate tickets in training and testing.

In [3]:
# Nombre de tickets avant nettoyage
before = len(df_clean)
# Supprimer les descriptions identiques
df_clean = df_clean.drop_duplicates(
    subset=["ticket_text"],
    keep="first"
).copy()
# Réinitialiser les index
df_clean = df_clean.reset_index(drop=True)

# Nombre de tickets après nettoyage
after = len(df_clean)
# Afficher les résultats
print("Tickets avant :", before)
print("Tickets après :", after)
print("Doublons supprimés :", before - after)

# Vérifier qu'il ne reste aucun doublon exact
print(
    "Doublons restants :",
    df_clean["ticket_text"].duplicated().sum()
)


Tickets avant : 4000
Tickets après : 3636
Doublons supprimés : 364
Doublons restants : 0



## 3. Near-Duplicate Detection
Identify highly similar ticket descriptions to reduce
the risk of data leakage during model evaluation.

Similar descriptions are not automatically removed.


In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Représentation TF-IDF des descriptions uniques
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+(?:[-_]\w+)*\b",
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df_clean["ticket_text"])

# Recherche du voisin le plus proche pour chaque ticket
nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine",
    algorithm="brute"
)

nn.fit(X_tfidf)

distances, indices = nn.kneighbors(X_tfidf)

# Similarité avec le voisin distinct le plus proche
nearest_similarity = 1 - distances[:, 1]

# Ajouter le résultat à la copie de travail
df_clean["nearest_similarity"] = nearest_similarity

# Afficher les statistiques
print("Tickets analysés :", len(df_clean))

print(
    "Similarité moyenne :",
    round(nearest_similarity.mean(), 4)
)

print(
    "Tickets avec similarité >= 0.90 :",
    (nearest_similarity >= 0.90).sum()
)

print(
    "Tickets avec similarité >= 0.80 :",
    (nearest_similarity >= 0.80).sum()
)

# Afficher quelques paires très similaires
print("\nExemples de quasi-doublons :")

top_indices = np.argsort(nearest_similarity)[-5:][::-1]

for i in top_indices:
    j = indices[i, 1]

    print("\nSimilarité :", round(nearest_similarity[i], 3))
    print("Ticket A :", df_clean.loc[i, "ticket_text"])
    print("Ticket B :", df_clean.loc[j, "ticket_text"])

# Représentation TF-IDF des descriptions uniques
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+(?:[-_]\w+)*\b",
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df_clean["ticket_text"])

# Recherche du voisin le plus proche pour chaque ticket
nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine",
    algorithm="brute"
)
nn.fit(X_tfidf)

distances, indices = nn.kneighbors(X_tfidf)
# Similarité avec le voisin distinct le plus proche
nearest_similarity = 1 - distances[:, 1]

# Ajouter le résultat à la copie de travail
df_clean["nearest_similarity"] = nearest_similarity

# Afficher les statistiques
print("Tickets analysés :", len(df_clean))

print(
    "Similarité moyenne :",
    round(nearest_similarity.mean(), 4)
)

print(
    "Tickets avec similarité >= 0.90 :",
    (nearest_similarity >= 0.90).sum()
)

print(
    "Tickets avec similarité >= 0.80 :",
    (nearest_similarity >= 0.80).sum()
)

# Afficher quelques paires très similaires
print("\nExemples de quasi-doublons :")

top_indices = np.argsort(nearest_similarity)[-5:][::-1]

for i in top_indices:
    j = indices[i, 1]

    print("\nSimilarité :", round(nearest_similarity[i], 3))
    print("Ticket A :", df_clean.loc[i, "ticket_text"])
    print("Ticket B :", df_clean.loc[j, "ticket_text"])


Tickets analysés : 3636
Similarité moyenne : 0.7263
Tickets avec similarité >= 0.90 : 280
Tickets avec similarité >= 0.80 : 1239

Exemples de quasi-doublons :

Similarité : 0.976
Ticket A : not urgent - forgot windows password and machine is now locked
Ticket B : urgent - forgot windows password and machine is now locked

Similarité : 0.976
Ticket A : urgent - forgot windows password and machine is now locked
Ticket B : not urgent - forgot windows password and machine is now locked

Similarité : 0.959
Ticket A : urgent - need to reset mfa, lost access to authenticator app after phone change
Ticket B : need to reset mfa, lost access to authenticator app after phone change

Similarité : 0.959
Ticket A : need to reset mfa, lost access to authenticator app after phone change
Ticket B : urgent - need to reset mfa, lost access to authenticator app after phone change

Similarité : 0.953
Ticket A : laptop randomly shuts off during meetings, very disruptive, kindly resolve this
Ticket B : i - l


### 3.2 Near-Duplicate Grouping

Group ticket descriptions with TF-IDF cosine similarity >= 0.80.

Connected descriptions receive the same group_id.

Objective: prevent near-duplicate leakage across
training, validation and test sets.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
import numpy as np

# Seuil de similarité
threshold = 0.80

# Transformer les descriptions en vecteurs TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+(?:[-_]\w+)*\b",
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df_clean["ticket_text"])

# Calculer la similarité entre toutes les descriptions
similarity_matrix = cosine_similarity(X)

# Construire les liens entre tickets similaires
adjacency = similarity_matrix >= threshold

# Supprimer les liens d'un ticket avec lui-même
np.fill_diagonal(adjacency, False)

# Identifier les groupes connectés
n_groups, group_ids = connected_components(
    csr_matrix(adjacency),
    directed=False
)

# Ajouter les groupes au dataset
df_clean["group_id"] = group_ids

# Statistiques
group_sizes = df_clean["group_id"].value_counts()

print("Tickets :", len(df_clean))
print("Nombre de groupes :", n_groups)
print("Groupes contenant plusieurs tickets :", (group_sizes > 1).sum())
print("Taille du plus grand groupe :", group_sizes.max())

# Vérifier quelques groupes
display(
    df_clean.loc[
        df_clean["group_id"].isin(
            group_sizes[group_sizes > 1].index[:3]
        ),
        ["ticket_text", "category", "group_id"]
    ].sort_values("group_id")
)


Tickets : 3636
Nombre de groupes : 2662
Groupes contenant plusieurs tickets : 265
Taille du plus grand groupe : 17


,ticket_text,category,group_id
31,"chrome not launching at all, task manager show...",Software,30
60,"chrome not launching at all, task manager show...",Software,30
129,"chrome not launching at all, task manager show...",Software,30
392,"chrome nto launching at all, task manager show...",Software,30
511,"my laptop - chrome not launching at all, task ...",Software,30
752,"urgent - chrome not launching at all, task man...",Software,30
848,"chrome not launching at all, task manager show...",Software,30
824,"chrome not launching at all, task manager show...",Software,30
1781,whole team blocked - chrome not launching at a...,Software,30
1820,"cchrome not launching at all, task manager sho...",Software,30


In [6]:
# Nombre de catégories différentes dans chaque groupe
categories_per_group = df_clean.groupby("group_id")["category"].nunique()
# Groupes contenant plusieurs catégories
mixed_groups = categories_per_group[categories_per_group > 1]
print("Groupes avec plusieurs catégories :", len(mixed_groups))


Groupes avec plusieurs catégories : 0
